In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import subprocess
subprocess.run(['pip', 'install', 'tqdm', '-q'])
print('Drive mounted. Ready.')

Mounted at /content/drive
Drive mounted. Ready.


In [ ]:
import os, glob, json, math, pickle
import xml.etree.ElementTree as ET
from collections import defaultdict
import numpy as np
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# ── PATHS ─────────────────────────────────────────────────────
PIE_PATH        = '/content/drive/MyDrive/PIE'
LANDMARK_PATH   = f'{PIE_PATH}/landmarks'
ANNOTATION_PATH = f'{PIE_PATH}/annotations'
ATTR_PATH       = f'{PIE_PATH}/annotations_attributes'
OUTPUT_PATH     = f'{PIE_PATH}/features'
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ── CONFIG ────────────────────────────────────────────────────
SEQ_LEN    = 30    # frames per LSTM sequence (1 second at 30fps)
SEQ_STEP   = 15    # sliding window stride (50% overlap)
WINDOW_SIZE = 10   # frames of history used per-feature calculation
FPS        = 30

# ── LABEL NAMES ───────────────────────────────────────────────
LABEL_NAMES = {0:'Waiting', 1:'Hesitant', 2:'Committed',
               3:'Distracted', 4:'Aggressive', 5:'Jaywalk'}

# ── MEDIAPIPE LANDMARK INDICES ────────────────────────────────
NOSE=0; L_EAR=7; R_EAR=8
L_SHOULDER=11; R_SHOULDER=12
L_ELBOW=13;    R_ELBOW=14
L_HIP=23;      R_HIP=24
L_KNEE=25;     R_KNEE=26
L_ANKLE=27;    R_ANKLE=28
L_HEEL=29;     R_HEEL=30
L_FOOT_IDX=31; R_FOOT_IDX=32

# ── FEATURE NAMES (for documentation) ────────────────────────
FEATURE_NAMES = [
    # Motion [0-9]
    'current_speed','avg_speed','max_speed','acceleration','deceleration',
    'speed_variance','step_frequency','left_step_length','right_step_length','pause_between_steps',
    # Pose [10-21]
    'upper_body_angle','lower_body_angle','head_angle','head_turn_frequency',
    'shoulder_angle','hip_angle','foot_angle_left','foot_angle_right',
    'forward_lean','lateral_lean','body_orientation','body_orientation_change',
    # Behavioral [22-30]
    'pause_duration','hesitation_cycles','total_hesitation_time',
    'distance_to_curb','distance_change_rate','temporal_movement_probability',
    'looks_left','looks_right','direction_changes',
    # Context [31-35]
    'traffic_light_state','vehicle_distance','crosswalk_presence',
    'road_width','pedestrian_density'
]

print('Configuration loaded.')
print(f'  Landmark path   : {LANDMARK_PATH}')
print(f'  Annotation path : {ANNOTATION_PATH}')
print(f'  Output path     : {OUTPUT_PATH}')
print(f'  Sequence length : {SEQ_LEN} frames ({SEQ_LEN/FPS:.1f}s)')

Configuration loaded.
  Landmark path   : /content/drive/MyDrive/PIE/landmarks
  Annotation path : /content/drive/MyDrive/PIE/annotations
  Output path     : /content/drive/MyDrive/PIE/features
  Sequence length : 30 frames (1.0s)


In [ ]:
def load_landmarks(npy_path):
    """Load .npy file → shape (33, 3): each row is [x, y, z] normalised 0-1."""
    return np.load(npy_path).reshape(33, 3)

def midpoint(a, b):
    return (np.array(a) + np.array(b)) / 2.0

def euclidean(a, b):
    return float(np.linalg.norm(np.array(a) - np.array(b)))

def angle_between(p1, vertex, p2):
    """Angle in degrees at vertex in triangle p1-vertex-p2."""
    v1 = np.array(p1) - np.array(vertex)
    v2 = np.array(p2) - np.array(vertex)
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 < 1e-6 or n2 < 1e-6:
        return 0.0
    return float(np.degrees(np.arccos(np.clip(np.dot(v1,v2)/(n1*n2), -1.0, 1.0))))

def vertical_angle(p1, p2):
    """Angle of vector (p1->p2) relative to vertical. Returns [-180, 180] degrees."""
    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]  # y increases downward in image
    return float(np.degrees(np.arctan2(dx, -dy)))

def horizontal_angle(p1, p2):
    """Angle of vector (p1->p2) relative to horizontal. Returns [-90, 90] degrees."""
    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]
    if abs(dx) < 1e-6:
        return 90.0
    return float(np.degrees(np.arctan(dy / dx)))

print('Geometry helpers defined.')

Geometry helpers defined.


In [ ]:
def parse_annotation_xml(xml_path):
    frame_data = defaultdict(lambda: {
        'pedestrians': [], 'traffic_lights': [], 'crosswalks': [], 'vehicles': []
    })
    try:
        root = ET.parse(xml_path).getroot()
    except Exception as e:
        print(f'  Warning: cannot parse {xml_path}: {e}')
        return frame_data

    for track in root.findall('track'):
        label = track.attrib.get('label', '')

        for box in track.findall('box'):
            # Skip outside frames
            if box.attrib.get('outside', '0') == '1':
                continue

            fid  = int(box.attrib['frame'])
            bbox = [float(box.attrib.get(k, 0)) for k in ('xtl', 'ytl', 'xbr', 'ybr')]
            occ_raw = box.attrib.get('occluded', '0')
            occ = int(occ_raw)

            # ── Read ALL attributes from inside the box ──────────
            attrs = {a.attrib['name']: (a.text or '').strip()
                     for a in box.findall('attribute')}

            if label == 'pedestrian':
                # ID is stored as an attribute INSIDE the box, e.g. '1_1_1'
                ped_id = attrs.get('id', '')

                # Map occlusion text to numeric (PIE uses 'none'/'part'/'full')
                occ_text = attrs.get('occlusion', 'none')
                occ_num = {'none': 0, 'part': 1, 'full': 2}.get(occ_text, occ)

                frame_data[fid]['pedestrians'].append({
                    'id':        ped_id,
                    'bbox':      bbox,
                    'occlusion': occ_num,
                    'action':    attrs.get('action',  'standing'),
                    'gesture':   attrs.get('gesture', 'none'),
                    'look':      attrs.get('look',    'not-looking'),
                    'cross':     attrs.get('cross',   'not-crossing')
                })

            elif label == 'traffic_light':
                attrs2 = {a.attrib['name']: (a.text or '').strip()
                          for a in box.findall('attribute')}
                frame_data[fid]['traffic_lights'].append({
                    'state': attrs2.get('state', 'none'),
                    'bbox':  bbox
                })

            elif label == 'crosswalk':
                frame_data[fid]['crosswalks'].append({'bbox': bbox})

            elif label == 'vehicle':
                attrs2 = {a.attrib['name']: (a.text or '').strip()
                          for a in box.findall('attribute')}
                frame_data[fid]['vehicles'].append({
                    'bbox': bbox,
                    'type': attrs2.get('type', 'car')
                })

    return frame_data

print('Fixed XML parser defined.')
print('Pedestrian ID now read from box attribute (e.g. 1_1_1) — will match attributes XML.')


def parse_attributes_xml(attr_path):
    """
    Parse annotations_attributes XML.
    Returns: { ped_id(str): {'intention_prob':float, 'crossing':int, 'num_lanes':int, ...} }
    """
    ped_attrs = {}
    if not attr_path or not os.path.exists(attr_path):
        return ped_attrs
    try:
        root = ET.parse(attr_path).getroot()
    except:
        return ped_attrs
    for ped in root.findall('.//pedestrian'):
        pid = ped.attrib.get('id', '')
        ped_attrs[pid] = {
            'intention_prob': float(ped.attrib.get('intention_prob', 0.5)),
            'crossing':       int(ped.attrib.get('crossing', 0)),
            'num_lanes':      int(ped.attrib.get('num_lanes', 2)),
            'signalized':     ped.attrib.get('signalized', 'n/a'),
            'crossing_point': int(ped.attrib.get('crossing_point', -1)),
            'critical_point': int(ped.attrib.get('critical_point', -1)),
        }
    return ped_attrs


print('XML parsers defined.')

Fixed XML parser defined.
Pedestrian ID now read from box attribute (e.g. 1_1_1) — will match attributes XML.
XML parsers defined.


In [ ]:
TL_MAP = {'red': 0.0, 'yellow': 0.5, 'green': 1.0, 'none': -1.0}

def bbox_center(bbox):
    x1,y1,x2,y2 = bbox
    return ((x1+x2)/2, (y1+y2)/2)

def bbox_iou(b1, b2):
    ix1=max(b1[0],b2[0]); iy1=max(b1[1],b2[1])
    ix2=min(b1[2],b2[2]); iy2=min(b1[3],b2[3])
    inter = max(0,ix2-ix1)*max(0,iy2-iy1)
    if inter == 0: return 0.0
    a1 = (b1[2]-b1[0])*(b1[3]-b1[1])
    a2 = (b2[2]-b2[0])*(b2[3]-b2[1])
    return inter/(a1+a2-inter+1e-6)

def ctx_traffic_light(frame_info, ped_bbox):
    """Encoded traffic light state: red=0, yellow=0.5, green=1, none=-1"""
    lights = frame_info.get('traffic_lights', [])
    if not lights: return -1.0
    pcx, pcy = bbox_center(ped_bbox)
    best_d, best_s = float('inf'), 'none'
    for tl in lights:
        cx,cy = bbox_center(tl['bbox'])
        d = math.hypot(cx-pcx, cy-pcy)
        if d < best_d: best_d, best_s = d, tl.get('state','none')
    return TL_MAP.get(best_s, -1.0)

def ctx_vehicle_distance(frame_info, ped_bbox):
    """Normalised distance to nearest vehicle (0=close, 1=far). Max reference: 1920px."""
    vehicles = frame_info.get('vehicles', [])
    if not vehicles: return 1.0
    pcx,pcy = bbox_center(ped_bbox)
    min_d = min(math.hypot(*[c-p for c,p in zip(bbox_center(v['bbox']),(pcx,pcy))]) for v in vehicles)
    return float(np.clip(min_d/1920.0, 0.0, 1.0))

def ctx_crosswalk(frame_info, ped_bbox):
    """1.0 if pedestrian bbox overlaps a crosswalk, else 0.0"""
    return 1.0 if any(bbox_iou(ped_bbox, cw['bbox']) > 0.01 for cw in frame_info.get('crosswalks',[])) else 0.0

def ctx_road_width(num_lanes):
    """Normalised road width from lane count (max 6 lanes)."""
    return float(np.clip(num_lanes/6.0, 0.0, 1.0))

def ctx_ped_density(frame_info, ped_id):
    """Number of other visible pedestrians, normalised by 10."""
    count = sum(1 for p in frame_info.get('pedestrians',[]) if p['id'] != ped_id)
    return float(np.clip(count/10.0, 0.0, 1.0))

print('Context helpers defined.')

Context helpers defined.


In [ ]:
def compute_36_features(lm_window, frame_info, ped_entry, num_lanes, history):
    """
    Compute all 36 features for the current frame.
    lm_window: list of (33,3) arrays, oldest first, current frame LAST
    Returns: float32 numpy array of shape (36,)
    """
    F = np.zeros(36, dtype=np.float32)
    lm = lm_window[-1]          # current frame (33,3) — normalised 0-1
    bbox = ped_entry['bbox']    # [x1,y1,x2,y2] in pixels
    ped_id = ped_entry['id']

    # Scale landmarks to bbox pixel coords
    bw = max(bbox[2]-bbox[0], 1.0)
    bh = max(bbox[3]-bbox[1], 1.0)

    def px(idx, lm_arr=None):
        arr = lm_arr if lm_arr is not None else lm
        return np.array([arr[idx,0]*bw, arr[idx,1]*bh])

    # ── Hip centroid & speed series ───────────────────────────
    hip_c = midpoint(px(L_HIP), px(R_HIP))
    speeds = []
    for i in range(1, len(lm_window)):
        h_p = midpoint(px(L_HIP, lm_window[i-1]), px(R_HIP, lm_window[i-1]))
        h_c = midpoint(px(L_HIP, lm_window[i]),   px(R_HIP, lm_window[i]))
        speeds.append(euclidean(h_c, h_p))
    if not speeds: speeds = [0.0]
    spd = np.array(speeds)

    # ═══ MOTION [0-9] ═══════════════════════════════════════════
    F[0] = spd[-1]                              # current_speed
    F[1] = float(spd.mean())                    # avg_speed
    F[2] = float(spd.max())                     # max_speed
    F[3] = max(spd[-1]-spd[-2], 0.0) if len(spd)>=2 else 0.0  # acceleration
    F[4] = max(spd[-2]-spd[-1], 0.0) if len(spd)>=2 else 0.0  # deceleration
    F[5] = float(spd.var())                     # speed_variance

    STEP_TH = 2.0  # px/frame threshold for a step event
    step_ev = 0; still_cnt = 0
    for i in range(1, len(lm_window)):
        la = px(L_ANKLE, lm_window[i]);   la_p = px(L_ANKLE, lm_window[i-1])
        ra = px(R_ANKLE, lm_window[i]);   ra_p = px(R_ANKLE, lm_window[i-1])
        if euclidean(la,la_p) > STEP_TH:  step_ev += 1
        if euclidean(la,la_p) < STEP_TH and euclidean(ra,ra_p) < STEP_TH: still_cnt += 1

    n_w = max(len(lm_window)-1, 1)
    F[6] = step_ev / n_w                        # step_frequency

    la_c = px(L_ANKLE); ra_c = px(R_ANKLE)
    la_p = px(L_ANKLE, lm_window[-2]) if len(lm_window)>=2 else la_c
    ra_p = px(R_ANKLE, lm_window[-2]) if len(lm_window)>=2 else ra_c
    F[7] = euclidean(la_c, la_p)                # left_step_length
    F[8] = euclidean(ra_c, ra_p)                # right_step_length
    F[9] = still_cnt / n_w                      # pause_between_steps

    # ═══ POSE [10-21] ════════════════════════════════════════════
    sl=px(L_SHOULDER); sr=px(R_SHOULDER)
    hl=px(L_HIP);      hr=px(R_HIP)
    kl=px(L_KNEE);     kr=px(R_KNEE)
    al=px(L_ANKLE);    ar=px(R_ANKLE)
    fl=px(L_FOOT_IDX); fr=px(R_FOOT_IDX)
    ns=px(NOSE)

    s_mid = midpoint(sl,sr)
    h_mid = midpoint(hl,hr)
    k_mid = midpoint(kl,kr)
    a_mid = midpoint(al,ar)

    F[10] = vertical_angle(h_mid, s_mid)                # upper_body_angle
    F[11] = angle_between(h_mid, k_mid, a_mid)          # lower_body_angle
    F[12] = vertical_angle(s_mid, ns)                   # head_angle

    # head_turn_frequency: lateral head direction changes across window
    nose_xs = [px(NOSE, lm_window[i])[0] for i in range(len(lm_window))]
    scx_avg = np.mean([(px(L_SHOULDER,lm_window[i])[0]+px(R_SHOULDER,lm_window[i])[0])/2
                        for i in range(len(lm_window))])
    sides = [1 if x > scx_avg else -1 for x in nose_xs]
    turns = sum(1 for i in range(1,len(sides)) if sides[i]!=sides[i-1])
    F[13] = turns / n_w                                  # head_turn_frequency

    F[14] = horizontal_angle(sl, sr)                     # shoulder_angle
    F[15] = horizontal_angle(hl, hr)                     # hip_angle
    F[16] = horizontal_angle(al, fl)                     # foot_angle_left
    F[17] = horizontal_angle(ar, fr)                     # foot_angle_right
    F[18] = abs(F[10])                                   # forward_lean
    F[19] = abs(s_mid[0] - h_mid[0])                    # lateral_lean
    F[20] = horizontal_angle(sl, sr)                     # body_orientation

    orients = [horizontal_angle(px(L_SHOULDER,lm_window[i]),
                                px(R_SHOULDER,lm_window[i]))
               for i in range(len(lm_window))]
    oc = [abs(orients[i]-orients[i-1]) for i in range(1,len(orients))]
    F[21] = float(np.mean(oc)) if oc else 0.0           # body_orientation_change

    # ═══ BEHAVIORAL [22-30] ══════════════════════════════════════
    SPEED_TH = 1.0  # px/frame
    is_still  = F[0] < SPEED_TH
    moving    = not is_still

    history['still']  = history.get('still', 0)  + (1 if is_still else 0)
    history['htime']  = history.get('htime', 0)  + (1 if not moving else 0)
    if moving != history.get('was_moving', False):
        history['hcycles'] = history.get('hcycles', 0) + 1
    history['was_moving'] = moving

    n_hist = max(history.get('total_frames', 1), 1)
    history['total_frames'] = history.get('total_frames', 0) + 1

    F[22] = history['still']   / n_hist              # pause_duration
    F[23] = history.get('hcycles',0) / 10.0          # hesitation_cycles
    F[24] = history['htime']   / n_hist              # total_hesitation_time

    # distance_to_curb: max ankle y in normalised coords (closer to 1 = bottom = near curb)
    F[25] = float(np.clip(max(lm[L_ANKLE,1], lm[R_ANKLE,1]), 0.0, 1.0))

    prev_ay = max(lm_window[-2][L_ANKLE,1], lm_window[-2][R_ANKLE,1]) if len(lm_window)>=2 else F[25]
    F[26] = float(F[25] - prev_ay)                   # distance_change_rate

    # temporal_movement_probability — from kinematics only (NO label leakage)
    p_spd = float(np.clip(F[1]/10.0,  0.0, 1.0))
    p_var = float(np.clip(F[5]/50.0,  0.0, 1.0))
    p_stp = float(np.clip(F[6],       0.0, 1.0))
    F[27] = float(np.clip((p_spd+p_var+p_stp)/3.0, 0.0, 1.0))

    # looks_left / looks_right: nose x vs shoulder midpoint x
    nose_x_n = lm[NOSE,0]
    scx_n    = (lm[L_SHOULDER,0]+lm[R_SHOULDER,0])/2.0
    F[28] = 1.0 if nose_x_n < scx_n else 0.0        # looks_left
    F[29] = 1.0 if nose_x_n > scx_n else 0.0        # looks_right

    # direction_changes: reversals in orientation delta sign
    dc = sum(1 for i in range(1,len(oc)) if (oc[i-1]>0)!=(oc[i]>0)) if len(oc)>1 else 0
    F[30] = dc / 10.0                                # direction_changes

    # ═══ CONTEXT [31-35] ═════════════════════════════════════════
    F[31] = ctx_traffic_light(frame_info, bbox)      # traffic_light_state
    F[32] = ctx_vehicle_distance(frame_info, bbox)   # vehicle_distance
    F[33] = ctx_crosswalk(frame_info, bbox)          # crosswalk_presence
    F[34] = ctx_road_width(num_lanes)                # road_width
    F[35] = ctx_ped_density(frame_info, ped_id)      # pedestrian_density

    return F

print('36-feature calculator defined.')

36-feature calculator defined.


In [ ]:
def assign_label(ped_entry, attrs, frame_info, speed_var):
    """
    Assign one of 6 mental state labels.
    Priority order: Jaywalk > Aggressive > Committed > Hesitant > Distracted > Waiting

    intention_prob is a PER-TRACK scalar from annotations_attributes XML.
    It is used ONLY as a label signal — never as a feature input.
    """
    intention = attrs.get('intention_prob', 0.5)
    action    = ped_entry.get('action',  'standing')
    gesture   = ped_entry.get('gesture', 'none')
    look      = ped_entry.get('look',    'not-looking')
    cross     = ped_entry.get('cross',   'not-crossing')
    crosswalk = ctx_crosswalk(frame_info, ped_entry['bbox'])

    # 5 — Jaywalk: crossing but NOT in a crosswalk
    if cross == 'crossing' and crosswalk == 0.0:
        return 5

    # 4 — Aggressive/Abnormal gait: hand gestures or very erratic speed
    if gesture in {'hand_yield','hand_rightofway','hand_ack'} or speed_var > 100.0:
        return 4

    # 2 — Committed: high intention AND actively crossing
    if intention > 0.66 and cross == 'crossing':
        return 2

    # 1 — Hesitant: mid-range intention (approach but uncertain)
    if 0.33 <= intention <= 0.66:
        return 1

    # 3 — Distracted: low intention AND not looking at road
    if intention < 0.33 and look == 'not-looking':
        return 3

    # 0 — Waiting: default (stationary, low intention)
    return 0

print('Label assignment defined.')
print('Label map:')
for k,v in LABEL_NAMES.items(): print(f'  {k} = {v}')

Label assignment defined.
Label map:
  0 = Waiting
  1 = Hesitant
  2 = Committed
  3 = Distracted
  4 = Aggressive
  5 = Jaywalk


In [ ]:
all_features = []  # list of (36,) arrays
all_labels   = []  # list of int labels
all_meta     = []  # list of (set_id, video_id, ped_id, frame_id)

SETS = ['set01','set02','set03','set04','set05','set06']

for set_id in SETS:
    lm_set_dir   = f'{LANDMARK_PATH}/{set_id}'
    ann_set_dir  = f'{ANNOTATION_PATH}/{set_id}'
    attr_set_dir = f'{ATTR_PATH}/{set_id}'

    if not os.path.isdir(lm_set_dir):
        print(f'Skipping {set_id} — no landmark folder'); continue

    ann_files = sorted(glob.glob(f'{ann_set_dir}/*_annt.xml'))
    if not ann_files:
        ann_files = sorted(glob.glob(f'{ann_set_dir}/*.xml'))

    print(f'\n{"="*55}')
    print(f'{set_id}: {len(ann_files)} annotation files found')
    print(f'{"="*55}')

    for ann_file in ann_files:
        base = os.path.basename(ann_file)

        # Derive video folder name from annotation filename
        # e.g. set01_video_0001_annt.xml → video_0001
        vid_folder = base.replace(f'{set_id}_','').replace('_annt.xml','').replace('.xml','')

        lm_vid_dir = f'{lm_set_dir}/{vid_folder}'
        if not os.path.isdir(lm_vid_dir):
            print(f'  Skip {base} — no landmark folder: {lm_vid_dir}'); continue

        # Find matching attributes file
        attr_file = f'{attr_set_dir}/{base.replace("_annt.xml","_attributes.xml")}'
        if not os.path.exists(attr_file):
            candidates = glob.glob(f'{attr_set_dir}/*{vid_folder}*')
            attr_file  = candidates[0] if candidates else None

        ped_attrs_map = parse_attributes_xml(attr_file)

        print(f'  Parsing: {base}')
        frame_data = parse_annotation_xml(ann_file)

        # Build frame_id → landmark path map
        lm_files = sorted(glob.glob(f'{lm_vid_dir}/frame_*.npy'))
        if not lm_files:
            print(f'    No .npy files in {lm_vid_dir}'); continue

        lm_map = {}
        for lf in lm_files:
            fid = int(os.path.basename(lf).replace('frame_','').replace('.npy',''))
            lm_map[fid] = lf

        # Group landmark frames by pedestrian track
        ped_track_frames = defaultdict(list)
        for fid in sorted(lm_map.keys()):
            if fid not in frame_data: continue
            # NEW — also skip __undefined__ gesture and blank IDs
            for ped in frame_data[fid]['pedestrians']:
                if ped['occlusion'] < 2 and ped['id'] != '':
                    ped_track_frames[ped['id']].append(fid)

        print(f'    {len(lm_map)} landmark frames | {len(ped_track_frames)} pedestrian tracks')

        for ped_id, track_frames in tqdm(ped_track_frames.items(),
                                          desc=f'    {vid_folder}', leave=False):
            track_frames = sorted(track_frames)
            attrs     = ped_attrs_map.get(ped_id, {'intention_prob':0.5,'num_lanes':2})
            num_lanes = attrs.get('num_lanes', 2)
            lm_window = []   # rolling window of landmark arrays
            history   = {}   # stateful behavioral counters

            for fid in track_frames:
                if fid not in lm_map: continue
                try:
                    lm = load_landmarks(lm_map[fid])  # (33,3)
                except:
                    continue

                lm_window.append(lm)
                if len(lm_window) > WINDOW_SIZE:
                    lm_window.pop(0)
                if len(lm_window) < 2:
                    continue  # need at least 2 frames for speed

                frame_info = frame_data.get(fid, {'pedestrians':[],'traffic_lights':[],'crosswalks':[],'vehicles':[]})
                ped_entry  = next((p for p in frame_info['pedestrians'] if p['id']==ped_id), None)
                if ped_entry is None: continue

                feats = compute_36_features(lm_window, frame_info, ped_entry, num_lanes, history)
                label = assign_label(ped_entry, attrs, frame_info, speed_var=float(feats[5]))

                all_features.append(feats)
                all_labels.append(label)
                all_meta.append((set_id, vid_folder, ped_id, fid))

print(f'\n{"="*55}')
print(f'EXTRACTION COMPLETE')
print(f'  Total frames : {len(all_features):,}')
print(f'{"="*55}')


set01: 4 annotation files found
  Parsing: video_0001_annt.xml
    86 landmark frames | 17 pedestrian tracks


  Parsing: video_0002_annt.xml
    1031 landmark frames | 35 pedestrian tracks


  Parsing: video_0003_annt.xml
    903 landmark frames | 44 pedestrian tracks


  Parsing: video_0004_annt.xml
    133 landmark frames | 8 pedestrian tracks



set02: 3 annotation files found
  Parsing: video_0001_annt.xml
    860 landmark frames | 33 pedestrian tracks


...

  Parsing: video_0007_annt.xml


    873 landmark frames | 15 pedestrian tracks


  Parsing: video_0008_annt.xml
    180 landmark frames | 13 pedestrian tracks


  Parsing: video_0009_annt.xml
    353 landmark frames | 29 pedestrian tracks



EXTRACTION COMPLETE
  Total frames : 82,791


In [ ]:
assert len(all_features) > 0, 'ERROR: No features extracted. Check paths and files.'

X = np.array(all_features, dtype=np.float32)   # (N, 36)
y = np.array(all_labels,   dtype=np.int32)     # (N,)

print(f'X shape : {X.shape}')
print(f'y shape : {y.shape}')

# Replace any NaN/Inf from edge-case geometry
bad = int(np.isnan(X).sum()) + int(np.isinf(X).sum())
if bad:
    print(f'  Fixing {bad} NaN/Inf values → 0')
    X = np.nan_to_num(X, nan=0.0, posinf=1.0, neginf=-1.0)

print()
print('Class distribution (per frame):')
for lid, lname in LABEL_NAMES.items():
    n   = int(np.sum(y==lid))
    pct = n/len(y)*100
    print(f'  {lid} {lname:<15} {n:>8,}  ({pct:5.1f}%)  {"█"*int(pct/2)}')

np.save(f'{OUTPUT_PATH}/X_features.npy', X)
np.save(f'{OUTPUT_PATH}/y_labels.npy',   y)
print(f'\nSaved X_features.npy  {X.shape}')
print(f'Saved y_labels.npy    {y.shape}')

X shape : (82791, 36)
y shape : (82791,)

Class distribution (per frame):
  0 Waiting           36,836  ( 44.5%)  ██████████████████████
  1 Hesitant           5,409  (  6.5%)  ███
  2 Committed         13,294  ( 16.1%)  ████████
  3 Distracted         6,595  (  8.0%)  ███
  4 Aggressive        14,153  ( 17.1%)  ████████
  5 Jaywalk            6,504  (  7.9%)  ███

Saved X_features.npy  (82791, 36)
Saved y_labels.npy    (82791,)


In [ ]:
# Fit scaler on training portion only (70%) to avoid data leakage
N = len(X)
scaler = StandardScaler()
scaler.fit(X[:int(N*0.70)])

X_scaled = scaler.transform(X).astype(np.float32)

with open(f'{OUTPUT_PATH}/feature_scaler.pkl','wb') as f:
    pickle.dump(scaler, f)

np.save(f'{OUTPUT_PATH}/X_features_scaled.npy', X_scaled)
print('Normalisation complete.')
print(f'Scaler mean[:5] : {scaler.mean_[:5].round(3)}')
print(f'Scaler std[:5]  : {scaler.scale_[:5].round(3)}')
print(f'Saved X_features_scaled.npy  {X_scaled.shape}')
print('Saved feature_scaler.pkl')

Normalisation complete.
Scaler mean[:5] : [ 6.482  6.562 19.117  2.927  2.926]
Scaler std[:5]  : [14.018  8.792 26.663 10.17  10.312]
Saved X_features_scaled.npy  (82791, 36)
Saved feature_scaler.pkl


In [ ]:
def make_sequences(X_sc, y_arr, meta, seq_len=30, step=15):
    """
    Build (S, 30, 36) sequences from per-frame data.
    Sequences only span frames from the SAME pedestrian track.
    Label = label of the LAST frame in each sequence.
    """
    # Group frame indices by (set_id, vid, ped_id)
    track_idx = defaultdict(list)
    for i, (sid,vid,pid,fid) in enumerate(meta):
        track_idx[(sid,vid,pid)].append(i)

    X_seq_l, y_seq_l = [], []
    for (sid,vid,pid), idxs in track_idx.items():
        idxs = sorted(idxs)
        if len(idxs) < seq_len: continue
        for start in range(0, len(idxs)-seq_len+1, step):
            win = idxs[start:start+seq_len]
            X_seq_l.append(X_sc[win])          # (30, 36)
            y_seq_l.append(int(y_arr[win[-1]])) # label of last frame

    return np.array(X_seq_l, dtype=np.float32), np.array(y_seq_l, dtype=np.int32)


print('Building sequences...')
X_seq, y_seq = make_sequences(X_scaled, y, all_meta, seq_len=SEQ_LEN, step=SEQ_STEP)

print(f'X_sequences : {X_seq.shape}   (sequences × frames × features)')
print(f'y_sequences : {y_seq.shape}')
print()
print('Sequence class distribution:')
for lid,lname in LABEL_NAMES.items():
    n   = int(np.sum(y_seq==lid))
    pct = n/len(y_seq)*100
    print(f'  {lid} {lname:<15} {n:>6,}  ({pct:5.1f}%)  {"█"*int(pct/2)}')

np.save(f'{OUTPUT_PATH}/X_sequences.npy', X_seq)
np.save(f'{OUTPUT_PATH}/y_sequences.npy', y_seq)
print(f'\nSaved X_sequences.npy  {X_seq.shape}')
print(f'Saved y_sequences.npy  {y_seq.shape}')

Building sequences...
X_sequences : (3888, 30, 36)   (sequences × frames × features)
y_sequences : (3888,)

Sequence class distribution:
  0 Waiting          1,714  ( 44.1%)  ██████████████████████
  1 Hesitant           216  (  5.6%)  ██
  2 Committed          785  ( 20.2%)  ██████████
  3 Distracted         218  (  5.6%)  ██
  4 Aggressive         616  ( 15.8%)  ███████
  5 Jaywalk            339  (  8.7%)  ████

Saved X_sequences.npy  (3888, 30, 36)
Saved y_sequences.npy  (3888,)


In [ ]:
idx = np.arange(len(X_seq))

# 70 / 15 / 15 stratified split
tr_idx, tmp_idx = train_test_split(idx, test_size=0.30, random_state=42, stratify=y_seq)
va_idx, te_idx  = train_test_split(tmp_idx, test_size=0.50, random_state=42, stratify=y_seq[tmp_idx])

X_train,y_train = X_seq[tr_idx], y_seq[tr_idx]
X_val,  y_val   = X_seq[va_idx], y_seq[va_idx]
X_test, y_test  = X_seq[te_idx], y_seq[te_idx]

total = len(X_seq)
print('Dataset splits:')
print(f'  Train : {X_train.shape}   ({len(X_train)/total*100:.0f}%)')
print(f'  Val   : {X_val.shape}     ({len(X_val)/total*100:.0f}%)')
print(f'  Test  : {X_test.shape}    ({len(X_test)/total*100:.0f}%)')

for tag,Xa,ya in [('train',X_train,y_train),('val',X_val,y_val),('test',X_test,y_test)]:
    np.save(f'{OUTPUT_PATH}/X_{tag}.npy', Xa)
    np.save(f'{OUTPUT_PATH}/y_{tag}.npy', ya)
    print(f'  Saved X_{tag}.npy, y_{tag}.npy')

Dataset splits:
  Train : (2721, 30, 36)   (70%)
  Val   : (583, 30, 36)     (15%)
  Test  : (584, 30, 36)    (15%)
  Saved X_train.npy, y_train.npy
  Saved X_val.npy, y_val.npy
  Saved X_test.npy, y_test.npy


In [ ]:
stats = {
    'total_frames':           int(len(X)),
    'total_sequences':        int(len(X_seq)),
    'seq_len':                SEQ_LEN,
    'seq_step':               SEQ_STEP,
    'feature_dim':            36,
    'feature_names':          FEATURE_NAMES,
    'label_names':            {str(k):v for k,v in LABEL_NAMES.items()},
    'class_counts_frames':    {LABEL_NAMES[i]: int(np.sum(y==i))     for i in range(6)},
    'class_counts_sequences': {LABEL_NAMES[i]: int(np.sum(y_seq==i)) for i in range(6)},
    'split_sizes':            {'train':int(len(X_train)),'val':int(len(X_val)),'test':int(len(X_test))},
    'feature_means':          [round(float(X[:,i].mean()),4) for i in range(36)],
    'feature_stds':           [round(float(X[:,i].std()), 4) for i in range(36)]
}

with open(f'{OUTPUT_PATH}/feature_stats.json','w') as f:
    json.dump(stats, f, indent=2)

print('='*55)
print('FEATURE EXTRACTION PIPELINE — COMPLETE')
print('='*55)
print(f'  Total frames     : {stats["total_frames"]:,}')
print(f'  Total sequences  : {stats["total_sequences"]:,}')
print(f'  Sequence shape   : ({SEQ_LEN}, 36)')
print(f'  Input to LSTM    : (batch, {SEQ_LEN}, 36)')
print()
print('Output files:')
for fname in ['X_features.npy','y_labels.npy','X_features_scaled.npy',
              'X_sequences.npy','y_sequences.npy',
              'X_train.npy','y_train.npy','X_val.npy','y_val.npy',
              'X_test.npy','y_test.npy','feature_scaler.pkl','feature_stats.json']:
    fp = f'{OUTPUT_PATH}/{fname}'
    if os.path.exists(fp):
        mb = os.path.getsize(fp)/1024/1024
        print(f'  ✅ {fname:<38} {mb:.1f} MB')
    else:
        print(f'  ❌ {fname}  NOT FOUND')

print()
print('  Load: X_train.npy, y_train.npy, X_val.npy, y_val.npy')
print('  Input shape: (batch_size, 30, 36)')
print('  Output classes: 6')

FEATURE EXTRACTION PIPELINE — COMPLETE
  Total frames     : 82,791
  Total sequences  : 3,888
  Sequence shape   : (30, 36)
  Input to LSTM    : (batch, 30, 36)

Output files:
  ✅ X_features.npy                         11.4 MB
  ✅ y_labels.npy                           0.3 MB
  ✅ X_features_scaled.npy                  11.4 MB
  ✅ X_sequences.npy                        16.0 MB
  ✅ y_sequences.npy                        0.0 MB
  ✅ X_train.npy                            11.2 MB
  ✅ y_train.npy                            0.0 MB
  ✅ X_val.npy                              2.4 MB
  ✅ y_val.npy                              0.0 MB
  ✅ X_test.npy                             2.4 MB
  ✅ y_test.npy                             0.0 MB
  ✅ feature_scaler.pkl                     0.0 MB
  ✅ feature_stats.json                     0.0 MB

Next step → lstm_training.ipynb
  Load: X_train.npy, y_train.npy, X_val.npy, y_val.npy
  Input shape: (batch_size, 30, 36)
  Output classes: 6


In [ ]:
X_check = np.load(f'{OUTPUT_PATH}/X_sequences.npy')
y_check = np.load(f'{OUTPUT_PATH}/y_sequences.npy')

i = 0
print(f'Sequence #{i}:')
print(f'  Shape : {X_check[i].shape}  (30 frames × 36 features)')
print(f'  Label : {y_check[i]} = {LABEL_NAMES[y_check[i]]}')
print()
print('First 5 frames, first 6 features:')
header = f'{"Frame":<7}' + ''.join(f'{FEATURE_NAMES[j][:12]:<14}' for j in range(6))
print('  '+header)
for fi in range(5):
    row = f'{fi:<7}' + ''.join(f'{X_check[i,fi,j]:+.4f}      ' for j in range(6))
    print('  '+row)

print()
print('Sanity check passed.')

Sequence #0:
  Shape : (30, 36)  (30 frames × 36 features)
  Label : 3 = Distracted

First 5 frames, first 6 features:
  Frame  current_spee  avg_speed     max_speed     acceleration  deceleration  speed_varian  
  0      +0.0117      +0.0095      -0.4677      -0.2878      -0.2837      -0.2325      
  1      -0.4079      -0.3249      -0.4677      -0.2878      +0.2866      -0.2166      
  2      -0.4553      -0.4616      -0.4677      -0.2878      -0.2192      -0.2166      
  3      -0.3932      -0.4836      -0.4422      -0.2023      -0.2837      -0.2169      
  4      -0.4463      -0.5262      -0.4358      -0.2878      -0.2097      -0.2181      

Sanity check passed. Ready for LSTM training.


In [ ]:
import numpy as np

# Load npy file
data = np.load("/content/drive/MyDrive/PIE/features/X_train.npy")

# Show basic info
print(type(data))
print(data.shape)

# Show first item
print(data[0])

<class 'numpy.ndarray'>
(2721, 30, 36)
[[-0.36540723 -0.67533886 -0.6659649  ... -0.6883357   0.76693946
   1.7424916 ]
 [-0.42569536 -0.67774564 -0.66572773 ... -0.6883357   0.76693946
   1.7424916 ]
 [-0.1744871  -0.6370062  -0.5655916  ... -0.6883357   0.76693946
   1.7424916 ]
 ...
 [-0.40420538 -0.67085564 -0.67273134 ...  1.4527797   0.76693946
   1.7424916 ]
 [-0.3553273  -0.6534686  -0.6606656  ...  1.4527797   0.76693946
   1.7424916 ]
 [-0.34602058 -0.64228606 -0.6557726  ...  1.4527797   0.76693946
   1.7424916 ]]


In [ ]:
import numpy as np

X = np.load("/content/drive/MyDrive/PIE/features/X_features.npy")
y = np.load("/content/drive/MyDrive/PIE/features/y_labels.npy")

# Feature index 5 = speed_variance
speed_var = X[:, 5]

print("Overall speed variance stats:")
print(f"  Mean : {speed_var.mean():.2f}")
print(f"  Std  : {speed_var.std():.2f}")
print(f"  Max  : {speed_var.max():.2f}")
print(f"  Min  : {speed_var.min():.2f}")

# Check per class
LABEL_NAMES = {0:'Waiting', 1:'Hesitant', 2:'Committed',
               3:'Distracted', 4:'Aggressive', 5:'Jaywalk'}

print("\nPer class speed variance:")
for lid, lname in LABEL_NAMES.items():
    mask = y == lid
    if mask.sum() > 0:
        print(f"  {lname:<15} mean={speed_var[mask].mean():.2f}  max={speed_var[mask].max():.2f}")

Overall speed variance stats:
  Mean : 134.82
  Std  : 538.16
  Max  : 24524.80
  Min  : 0.00

Per class speed variance:
  Waiting         mean=10.97  max=99.94
  Hesitant        mean=11.61  max=99.34
  Committed       mean=19.57  max=99.99
  Distracted      mean=14.72  max=100.00
  Aggressive      mean=653.45  max=24524.80
  Jaywalk         mean=167.46  max=15532.48


In [ ]:
import glob, numpy as np

PIE_PATH = "/content/drive/MyDrive/PIE"

# Count .npy landmark files
lm_files = glob.glob(f"{PIE_PATH}/landmarks/**/*.npy", recursive=True)
print(f"Total landmark .npy files : {len(lm_files):,}")

# Sample: how many are non-zero (pose detected)?
sample = lm_files[:500]
detected = sum(1 for f in sample if np.any(np.load(f) != 0))
print(f"Pose detected (sample 500): {detected}/500 = {detected/5:.1f}%")

# Count frames vs landmark files
img_files = glob.glob(f"{PIE_PATH}/images_annotated/**/*.jpg", recursive=True)
print(f"Total extracted frames     : {len(img_files):,}")
print(f"Landmark coverage          : {len(lm_files)/len(img_files)*100:.1f}%")

Total landmark .npy files : 33,279
Pose detected (sample 500): 500/500 = 100.0%
Total extracted frames     : 285,437
Landmark coverage          : 11.7%


In [1]:
import os, json
import xml.etree.ElementTree as ET

ATTR_PATH = '/content/drive/MyDrive/PIE/annotations_attributes'

# Look at one attributes file to find intention field
attr_files = []
for root, dirs, files in os.walk(ATTR_PATH):
    for f in files:
        if f.endswith('.xml') or f.endswith('.json'):
            attr_files.append(os.path.join(root, f))

print(f'Found {len(attr_files)} attribute files')
print(f'First file: {attr_files[0]}')

# Read and print first file to see structure
with open(attr_files[0], 'r') as f:
    content = f.read()
print(content[:2000])

Found 53 attribute files
First file: /content/drive/MyDrive/PIE/annotations_attributes/set01/video_0001_attributes.xml
<ped_attributes><pedestrian age="adult" critical_point="1595" crossing="0" crossing_point="1613" exp_start_point="1568" gender="male" id="1_1_7" intention_prob="0.8666666667" intersection="T-right" num_lanes="4" signalized="n/a" traffic_direction="TW" /><pedestrian age="adult" critical_point="15268" crossing="0" crossing_point="15298" exp_start_point="15180" gender="male" id="1_1_9" intention_prob="0.9166666667" intersection="four-way" num_lanes="4" signalized="CS" traffic_direction="TW" /><pedestrian age="adult" critical_point="16543" crossing="0" crossing_point="16561" exp_start_point="16453" gender="male" id="1_1_11" intention_prob="0.95" intersection="four-way" num_lanes="4" signalized="CS" traffic_direction="TW" /><pedestrian age="child" critical_point="1470" crossing="0" crossing_point="1516" exp_start_point="1406" gender="female" id="1_1_6" intention_prob="0.716

In [2]:
import numpy as np

OUTPUT_PATH = '/content/drive/MyDrive/PIE/features'

# Check if sequence metadata was saved
import os
files = os.listdir(OUTPUT_PATH)
print('Files in features folder:')
for f in sorted(files):
    print(f'  {f}')

Files in features folder:
  X_features.npy
  X_features_scaled.npy
  X_sequences.npy
  X_test.npy
  X_train.npy
  X_val.npy
  feature_scaler.pkl
  feature_stats.json
  y_labels.npy
  y_sequences.npy
  y_test.npy
  y_train.npy
  y_val.npy


Feature extraction with pedestrains

In [ ]:
# -*- coding: utf-8 -*-
"""
feature_extraction.py
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Full Feature Extraction Pipeline
Project: Proactive Social Compliance Modeling using Agentic Theory of Mind

KEY CHANGE vs original:
  Now saves sequence_ped_ids.npy alongside every sequence.
  This allows intention_prob to be correctly matched later.

OUTPUT FILES:
  X_features.npy         — (N_frames, 36)     all frame features
  y_labels.npy           — (N_frames,)         mental state per frame
  X_features_scaled.npy  — (N_frames, 36)     scaled features
  X_sequences.npy        — (N_seq, 30, 36)    30-frame windows
  y_sequences.npy        — (N_seq,)            mental state per sequence
  sequence_ped_ids.npy   — (N_seq,)   ← NEW   pedestrian ID per sequence
  X_train/val/test.npy
  y_train/val/test.npy
  feature_scaler.pkl
  feature_stats.json
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

# ── Cell 1: Mount Drive ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import subprocess
subprocess.run(['pip', 'install', 'tqdm', '-q'])
print('Drive mounted. Ready.')

# ── Cell 2: Config ─────────────────────────────────────────────
import os, glob, json, math, pickle
import xml.etree.ElementTree as ET
from collections import defaultdict
import numpy as np
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

PIE_PATH        = '/content/drive/MyDrive/PIE'
LANDMARK_PATH   = f'{PIE_PATH}/landmarks'
ANNOTATION_PATH = f'{PIE_PATH}/annotations'
ATTR_PATH       = f'{PIE_PATH}/annotations_attributes'
OUTPUT_PATH     = f'{PIE_PATH}/features_with_ped_id'
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEQ_LEN     = 30
SEQ_STEP    = 15
WINDOW_SIZE = 10
FPS         = 30

LABEL_NAMES = {
    0:'Waiting', 1:'Hesitant', 2:'Committed',
    3:'Distracted', 4:'Aggressive', 5:'Jaywalk'
}

NOSE=0; L_EAR=7; R_EAR=8
L_SHOULDER=11; R_SHOULDER=12
L_ELBOW=13;    R_ELBOW=14
L_HIP=23;      R_HIP=24
L_KNEE=25;     R_KNEE=26
L_ANKLE=27;    R_ANKLE=28
L_HEEL=29;     R_HEEL=30
L_FOOT_IDX=31; R_FOOT_IDX=32

FEATURE_NAMES = [
    'current_speed','avg_speed','max_speed','acceleration','deceleration',
    'speed_variance','step_frequency','left_step_length','right_step_length',
    'pause_between_steps',
    'upper_body_angle','lower_body_angle','head_angle','head_turn_frequency',
    'shoulder_angle','hip_angle','foot_angle_left','foot_angle_right',
    'forward_lean','lateral_lean','body_orientation','body_orientation_change',
    'pause_duration','hesitation_cycles','total_hesitation_time',
    'distance_to_curb','distance_change_rate','temporal_movement_probability',
    'looks_left','looks_right','direction_changes',
    'traffic_light_state','vehicle_distance','crosswalk_presence',
    'road_width','pedestrian_density'
]

print('Configuration loaded.')
print(f'  Sequence length : {SEQ_LEN} frames ({SEQ_LEN/FPS:.1f}s)')

# ── Cell 3: Geometry helpers ───────────────────────────────────
def load_landmarks(npy_path):
    return np.load(npy_path).reshape(33, 3)

def midpoint(a, b):
    return (np.array(a) + np.array(b)) / 2.0

def euclidean(a, b):
    return float(np.linalg.norm(np.array(a) - np.array(b)))

def angle_between(p1, vertex, p2):
    v1 = np.array(p1) - np.array(vertex)
    v2 = np.array(p2) - np.array(vertex)
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 < 1e-6 or n2 < 1e-6:
        return 0.0
    return float(np.degrees(
        np.arccos(np.clip(np.dot(v1,v2)/(n1*n2), -1.0, 1.0))
    ))

def vertical_angle(p1, p2):
    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]
    return float(np.degrees(np.arctan2(dx, -dy)))

def horizontal_angle(p1, p2):
    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]
    if abs(dx) < 1e-6:
        return 90.0
    return float(np.degrees(np.arctan(dy / dx)))

print('Geometry helpers defined.')

# ── Cell 4: Annotation parser ──────────────────────────────────
def parse_annotation_xml(xml_path):
    frame_data = defaultdict(lambda: {
        'pedestrians': [], 'traffic_lights': [],
        'crosswalks':  [], 'vehicles': []
    })
    try:
        root = ET.parse(xml_path).getroot()
    except Exception as e:
        print(f'  Warning: cannot parse {xml_path}: {e}')
        return frame_data

    for track in root.findall('track'):
        label = track.attrib.get('label', '')

        for box in track.findall('box'):
            if box.attrib.get('outside', '0') == '1':
                continue

            fid     = int(box.attrib['frame'])
            bbox    = [float(box.attrib.get(k, 0))
                       for k in ('xtl','ytl','xbr','ybr')]
            occ_raw = box.attrib.get('occluded', '0')
            occ     = int(occ_raw)
            attrs   = {a.attrib['name']: (a.text or '').strip()
                       for a in box.findall('attribute')}

            if label == 'pedestrian':
                ped_id   = attrs.get('id', '')
                occ_text = attrs.get('occlusion', 'none')
                occ_num  = {'none':0,'part':1,'full':2}.get(occ_text, occ)
                frame_data[fid]['pedestrians'].append({
                    'id':        ped_id,
                    'bbox':      bbox,
                    'occlusion': occ_num,
                    'action':    attrs.get('action',  'standing'),
                    'gesture':   attrs.get('gesture', 'none'),
                    'look':      attrs.get('look',    'not-looking'),
                    'cross':     attrs.get('cross',   'not-crossing'),
                })

            elif label == 'traffic_light':
                state_map = {'red':0.0, 'yellow':0.5, 'green':1.0}
                state_str = attrs.get('state', 'none').lower()
                frame_data[fid]['traffic_lights'].append(
                    state_map.get(state_str, -1.0)
                )

            elif label == 'crosswalk':
                frame_data[fid]['crosswalks'].append(bbox)

            elif label in ('car','bus','truck','vehicle'):
                frame_data[fid]['vehicles'].append(bbox)

    return frame_data

print('Annotation parser defined.')

# ── Cell 5: Mental state classifier ───────────────────────────
def classify_mental_state(action, gesture, look, cross,
                           speed, speed_var, dist_change):
    cross_lower  = cross.lower()  if cross  else ''
    action_lower = action.lower() if action else ''
    look_lower   = look.lower()   if look   else ''

    is_crossing   = 'crossing' in cross_lower
    is_not_cross  = 'not-crossing' in cross_lower or cross_lower == ''
    is_looking    = 'looking' in look_lower and 'not' not in look_lower
    is_moving     = speed > 0.5
    is_fast       = speed > 8.0
    is_high_var   = speed_var > 150
    toward_road   = dist_change > 0.01

    if is_crossing and is_fast and is_high_var:
        return 5    # Jaywalk

    if is_crossing and is_fast:
        return 4    # Aggressive

    if is_crossing and is_moving:
        return 2    # Committed

    if not is_moving and not is_crossing:
        return 0    # Waiting

    if toward_road and is_looking and not is_crossing:
        return 1    # Hesitant

    if is_moving and not is_looking and not is_crossing:
        return 3    # Distracted

    if toward_road and is_moving:
        return 1    # Hesitant

    return 0        # default Waiting

print('Mental state classifier defined.')

# ── Cell 6: Feature extractor ──────────────────────────────────
def extract_features_for_pedestrian(ped_id, frame_ids,
                                    landmark_dir, frame_data):
    """
    Extract 36 features for each frame of one pedestrian track.
    Returns list of feature vectors.
    """
    features_list = []
    positions     = []   # (x, y) centre positions over time

    for i, fid in enumerate(frame_ids):
        lm_path = os.path.join(landmark_dir, f'frame_{fid:05d}.npy')
        if not os.path.exists(lm_path):
            continue

        lm = load_landmarks(lm_path)   # (33, 3)

        # ── Skeleton keypoints ─────────────────────────────────
        nose       = lm[NOSE, :2]
        l_shoulder = lm[L_SHOULDER, :2]
        r_shoulder = lm[R_SHOULDER, :2]
        l_hip      = lm[L_HIP, :2]
        r_hip      = lm[R_HIP, :2]
        l_ankle    = lm[L_ANKLE, :2]
        r_ankle    = lm[R_ANKLE, :2]
        l_knee     = lm[L_KNEE, :2]
        r_knee     = lm[R_KNEE, :2]
        l_heel     = lm[L_HEEL, :2]
        r_heel     = lm[R_HEEL, :2]
        l_foot     = lm[L_FOOT_IDX, :2]
        r_foot     = lm[R_FOOT_IDX, :2]
        l_ear      = lm[L_EAR, :2]
        r_ear      = lm[R_EAR, :2]

        mid_shoulder = midpoint(l_shoulder, r_shoulder)
        mid_hip      = midpoint(l_hip,      r_hip)
        mid_ankle    = midpoint(l_ankle,    r_ankle)

        # Body centre = midpoint of hips
        cx, cy = float(mid_hip[0]), float(mid_hip[1])
        positions.append((cx, cy))

        # ── Window history ─────────────────────────────────────
        win_start   = max(0, len(positions) - WINDOW_SIZE)
        win_pos     = positions[win_start:]
        win_n       = len(win_pos)

        # Motion features
        if win_n >= 2:
            dists = [euclidean(win_pos[j], win_pos[j-1])
                     for j in range(1, win_n)]
            speeds        = [d * FPS for d in dists]
            current_speed = speeds[-1]
            avg_speed     = float(np.mean(speeds))
            max_speed     = float(np.max(speeds))
            accels        = [speeds[j]-speeds[j-1]
                             for j in range(1, len(speeds))]
            acceleration  = float(np.mean([a for a in accels if a > 0])) \
                            if any(a > 0 for a in accels) else 0.0
            deceleration  = float(np.mean([abs(a) for a in accels if a < 0])) \
                            if any(a < 0 for a in accels) else 0.0
            speed_var     = float(np.var(speeds))
            dist_change   = float(euclidean(win_pos[-1], win_pos[0]))
            dist_change_r = (dist_change / win_n) * np.sign(
                win_pos[-1][1] - win_pos[0][1]
            )
        else:
            current_speed = avg_speed = max_speed = 0.0
            acceleration  = deceleration = speed_var = 0.0
            dist_change_r = 0.0

        # Step features
        l_ankle_pos = [(lm[L_ANKLE,0], lm[L_ANKLE,1])]
        r_ankle_pos = [(lm[R_ANKLE,0], lm[R_ANKLE,1])]
        step_freq   = current_speed / 0.7 if current_speed > 0.1 else 0.0
        l_step_len  = euclidean(l_heel, l_foot)
        r_step_len  = euclidean(r_heel, r_foot)
        pause_steps = 1.0 if current_speed < 0.1 else 0.0

        # Pause / hesitation
        pause_dur   = float(sum(1 for s in (speeds if win_n>=2 else [0])
                                if s < 0.5)) / FPS
        hesit_cyc   = float(sum(
            1 for j in range(1, len(speeds))
            if speeds[j] < 0.5 < speeds[j-1]
        )) if win_n >= 2 else 0.0
        total_hesit = pause_dur

        # Movement probability
        move_prob = float(sum(1 for s in (speeds if win_n>=2 else [0])
                              if s > 0.5)) / max(win_n-1, 1)

        # ── Pose features ──────────────────────────────────────
        upper_body_ang  = vertical_angle(mid_hip, mid_shoulder)
        lower_body_ang  = vertical_angle(mid_ankle, mid_hip)
        head_ang        = vertical_angle(mid_shoulder, nose)
        shoulder_ang    = horizontal_angle(l_shoulder, r_shoulder)
        hip_ang         = horizontal_angle(l_hip, r_hip)
        foot_ang_l      = vertical_angle(l_knee, l_ankle)
        foot_ang_r      = vertical_angle(r_knee, r_ankle)
        forward_lean    = upper_body_ang
        lateral_lean    = shoulder_ang
        body_orient     = float(np.degrees(np.arctan2(
            mid_shoulder[0] - mid_hip[0],
            mid_hip[1]      - mid_shoulder[1]
        )))

        if len(positions) >= 2:
            prev = positions[-2]
            body_orient_chg = float(abs(
                np.degrees(np.arctan2(cx-prev[0], prev[1]-cy)) - body_orient
            ))
        else:
            body_orient_chg = 0.0

        # Head turns
        head_turn_freq = 0.0
        ear_diff       = abs(float(l_ear[0]) - float(r_ear[0]))
        if ear_diff > 0.05:
            head_turn_freq = min(ear_diff * 2.0, 1.0)

        # ── Context from annotation ────────────────────────────
        finfo        = frame_data.get(fid, {})
        tl_states    = finfo.get('traffic_lights', [])
        traffic_light= float(np.mean(tl_states)) if tl_states else -1.0

        crosswalks   = finfo.get('crosswalks', [])
        crosswalk    = 1.0 if crosswalks else 0.0

        vehicles     = finfo.get('vehicles', [])
        veh_dist     = 1.0
        if vehicles:
            dists = []
            for v in vehicles:
                vx = (v[0]+v[2])/2; vy = (v[1]+v[3])/2
                dists.append(euclidean((cx,cy),(vx/1920,vy/1080)))
            veh_dist = float(min(dists))
        veh_dist = float(np.clip(veh_dist, 0.0, 1.0))

        # Road width (normalised by frame)
        road_width = 1.0

        # Pedestrian density
        peds       = finfo.get('pedestrians', [])
        ped_density= min(len(peds) / 10.0, 1.0)

        # Distance to curb (approximation)
        dist_to_curb = float(np.clip(1.0 - cy, 0.0, 1.0))

        # Look left / look right
        head_yaw   = float(l_ear[0] - r_ear[0])
        looks_left = 1.0 if head_yaw >  0.03 else 0.0
        looks_right= 1.0 if head_yaw < -0.03 else 0.0

        # Direction changes
        dir_changes = 0.0
        if win_n >= 3:
            dirs = [np.sign(win_pos[j][0]-win_pos[j-1][0])
                    for j in range(1, win_n)]
            dir_changes = float(sum(
                1 for j in range(1, len(dirs)) if dirs[j] != dirs[j-1]
            ))

        # ── Assemble 36-feature vector ─────────────────────────
        F = np.array([
            current_speed, avg_speed, max_speed,
            acceleration, deceleration, speed_var,
            step_freq, l_step_len, r_step_len, pause_steps,
            upper_body_ang, lower_body_ang, head_ang, head_turn_freq,
            shoulder_ang, hip_ang, foot_ang_l, foot_ang_r,
            forward_lean, lateral_lean, body_orient, body_orient_chg,
            pause_dur, hesit_cyc, total_hesit,
            dist_to_curb, dist_change_r, move_prob,
            looks_left, looks_right, dir_changes,
            traffic_light, veh_dist, crosswalk,
            road_width, ped_density,
        ], dtype=np.float32)

        # Get action/look/cross from annotation for mental state
        action_str = 'standing'
        look_str   = 'not-looking'
        cross_str  = 'not-crossing'
        for ped in finfo.get('pedestrians', []):
            if ped.get('id','') == ped_id:
                action_str = ped.get('action',  'standing')
                look_str   = ped.get('look',    'not-looking')
                cross_str  = ped.get('cross',   'not-crossing')
                break

        mental_state = classify_mental_state(
            action_str, '', look_str, cross_str,
            current_speed, speed_var, dist_change_r
        )

        features_list.append((F, mental_state))

    return features_list

print('Feature extractor defined.')

# ── Cell 7: MAIN EXTRACTION LOOP ──────────────────────────────
print('\n' + '='*60)
print('STARTING FEATURE EXTRACTION')
print('='*60)

all_features   = []   # list of np.array (36,)
all_labels     = []   # mental state int
all_sequences  = []   # list of np.array (30, 36)
all_seq_labels = []   # mental state per sequence
all_seq_ped_ids= []   # ← NEW: pedestrian ID per sequence

ann_files = sorted(glob.glob(
    f'{ANNOTATION_PATH}/**/*.xml', recursive=True
))
print(f'Found {len(ann_files)} annotation files')

for ann_idx, ann_file in enumerate(tqdm(ann_files, desc='Processing')):

    parts    = ann_file.replace('\\','/').split('/')
    set_name = [p for p in parts if p.startswith('set')][0]
    vid_name = parts[-1].replace('_annt.xml','')

    lm_dir   = f'{LANDMARK_PATH}/{set_name}/{vid_name}'
    if not os.path.isdir(lm_dir):
        continue

    frame_data = parse_annotation_xml(ann_file)

    # ── Process each pedestrian track ─────────────────────────
    try:
        root = ET.parse(ann_file).getroot()
    except:
        continue

    for track in root.findall('track'):
        if track.attrib.get('label','') != 'pedestrian':
            continue

        # Get pedestrian ID from first box
        ped_id = ''
        for box in track.findall('box'):
            attrs  = {a.attrib['name']: (a.text or '').strip()
                      for a in box.findall('attribute')}
            ped_id = attrs.get('id','')
            if ped_id:
                break
        if not ped_id:
            continue

        # Collect valid frame IDs (have landmark file)
        frame_ids = []
        for box in track.findall('box'):
            if box.attrib.get('outside','0') == '1':
                continue
            fid     = int(box.attrib['frame'])
            lm_path = os.path.join(lm_dir, f'frame_{fid:05d}.npy')
            if os.path.exists(lm_path):
                frame_ids.append(fid)

        frame_ids = sorted(frame_ids)
        if len(frame_ids) < SEQ_LEN:
            continue

        # Extract features for all frames of this pedestrian
        ped_features = extract_features_for_pedestrian(
            ped_id, frame_ids, lm_dir, frame_data
        )

        if len(ped_features) < SEQ_LEN:
            continue

        # Store frame-level features
        for F, ms in ped_features:
            all_features.append(F)
            all_labels.append(ms)

        # ── Sliding window → sequences ─────────────────────────
        feat_array = np.array([f for f, _ in ped_features])  # (N, 36)
        ms_array   = np.array([ms for _, ms in ped_features]) # (N,)

        for start in range(0, len(feat_array) - SEQ_LEN + 1, SEQ_STEP):
            window     = feat_array[start:start + SEQ_LEN]    # (30, 36)
            window_ms  = ms_array[start:start + SEQ_LEN]
            seq_label  = int(np.bincount(window_ms).argmax()) # majority vote

            all_sequences.append(window)
            all_seq_labels.append(seq_label)
            all_seq_ped_ids.append(ped_id)   # ← NEW: save ped_id

print(f'\nExtraction complete.')
print(f'  Total frames     : {len(all_features):,}')
print(f'  Total sequences  : {len(all_sequences):,}')
print(f'  Sequence shape   : {all_sequences[0].shape}')

# ── Cell 8: Convert and scale ──────────────────────────────────
X_features   = np.array(all_features,   dtype=np.float32)
y_labels     = np.array(all_labels,     dtype=np.int32)
X_sequences  = np.array(all_sequences,  dtype=np.float32)
y_sequences  = np.array(all_seq_labels, dtype=np.int32)
seq_ped_ids  = np.array(all_seq_ped_ids)   # ← NEW

print(f'\nArrays created:')
print(f'  X_features  : {X_features.shape}')
print(f'  X_sequences : {X_sequences.shape}')
print(f'  y_sequences : {y_sequences.shape}')
print(f'  seq_ped_ids : {seq_ped_ids.shape}')   # ← NEW

# Scale frame-level features
scaler           = StandardScaler()
X_features_scaled= scaler.fit_transform(X_features)

# Scale sequences using same scaler
N, T, F_dim     = X_sequences.shape
X_seq_2d        = X_sequences.reshape(-1, F_dim)
X_seq_scaled_2d = scaler.transform(X_seq_2d)
X_seq_scaled    = X_seq_scaled_2d.reshape(N, T, F_dim)

print('\nScaling done.')

# ── Cell 9: Train/Val/Test split ───────────────────────────────
idx = np.arange(len(X_seq_scaled))

idx_trainval, idx_test = train_test_split(
    idx, test_size=0.2, random_state=42, stratify=y_sequences
)
idx_train, idx_val = train_test_split(
    idx_trainval, test_size=0.125, random_state=42,
    stratify=y_sequences[idx_trainval]
)

X_train = X_seq_scaled[idx_train]
X_val   = X_seq_scaled[idx_val]
X_test  = X_seq_scaled[idx_test]
y_train = y_sequences[idx_train]
y_val   = y_sequences[idx_val]
y_test  = y_sequences[idx_test]

print(f'Split:')
print(f'  Train : {X_train.shape}')
print(f'  Val   : {X_val.shape}')
print(f'  Test  : {X_test.shape}')

# Label distribution
for cls in range(6):
    n = (y_train == cls).sum()
    print(f'  {LABEL_NAMES[cls]:<14} train={n}')

# ── Cell 10: Save all files ────────────────────────────────────
print('\nSaving files...')

# Frame-level
np.save(f'{OUTPUT_PATH}/X_features.npy',        X_features)
np.save(f'{OUTPUT_PATH}/y_labels.npy',           y_labels)
np.save(f'{OUTPUT_PATH}/X_features_scaled.npy',  X_features_scaled)

# Sequence-level
np.save(f'{OUTPUT_PATH}/X_sequences.npy',        X_seq_scaled)
np.save(f'{OUTPUT_PATH}/y_sequences.npy',        y_sequences)
np.save(f'{OUTPUT_PATH}/sequence_ped_ids.npy',   seq_ped_ids)  # ← NEW

# Train/val/test
np.save(f'{OUTPUT_PATH}/X_train.npy', X_train)
np.save(f'{OUTPUT_PATH}/X_val.npy',   X_val)
np.save(f'{OUTPUT_PATH}/X_test.npy',  X_test)
np.save(f'{OUTPUT_PATH}/y_train.npy', y_train)
np.save(f'{OUTPUT_PATH}/y_val.npy',   y_val)
np.save(f'{OUTPUT_PATH}/y_test.npy',  y_test)

# Scaler
import joblib
joblib.dump(scaler, f'{OUTPUT_PATH}/feature_scaler.pkl')

# Feature stats
stats = {
    'n_features':   int(F_dim),
    'n_frames':     int(len(X_features)),
    'n_sequences':  int(len(X_sequences)),
    'seq_len':      int(SEQ_LEN),
    'seq_step':     int(SEQ_STEP),
    'feature_names': FEATURE_NAMES,
    'label_names':   LABEL_NAMES,
    'split': {
        'train': int(len(X_train)),
        'val':   int(len(X_val)),
        'test':  int(len(X_test)),
    }
}
with open(f'{OUTPUT_PATH}/feature_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print('\n' + '='*55)
print('FEATURE EXTRACTION PIPELINE — COMPLETE')
print('='*55)
print(f'  Total frames     : {len(X_features):,}')
print(f'  Total sequences  : {len(X_sequences):,}')
print(f'  Sequence shape   : (30, 36)')

print('\nOutput files:')
for fname in [
    'X_features.npy', 'y_labels.npy', 'X_features_scaled.npy',
    'X_sequences.npy', 'y_sequences.npy',
    'sequence_ped_ids.npy',   # ← NEW
    'X_train.npy', 'y_train.npy',
    'X_val.npy',   'y_val.npy',
    'X_test.npy',  'y_test.npy',
    'feature_scaler.pkl', 'feature_stats.json',
]:
    fp = f'{OUTPUT_PATH}/{fname}'
    if os.path.exists(fp):
        mb = os.path.getsize(fp)/1024/1024
        print(f'  ✅ {fname:<38} {mb:.1f} MB')
    else:
        print(f'  ❌ {fname}  NOT FOUND')

print('\nNext step → lstm_training.ipynb')
print('  Load: X_train.npy, y_train.npy, X_val.npy, y_val.npy')
print('  sequence_ped_ids.npy → use for intention_prob matching')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Ready.
Configuration loaded.
  Sequence length : 30 frames (1.0s)
Geometry helpers defined.
Annotation parser defined.
Mental state classifier defined.
Feature extractor defined.

STARTING FEATURE EXTRACTION
Found 53 annotation files


Processing:  62%|██████▏   | 33/53 [1:18:36<1:08:43, 206.15s/it]